In [12]:
import sys
sys.path.append("..")  # Adds the parent directory (src) to the Python path

In [13]:
import torch
torch.cuda.is_available()

True

Add scream transcripts

In [14]:
audio_path = '../media/lrntech_63f9fe069cef060011647ceb.wav'

In [ ]:
import torch
import numpy as np
import librosa
from transformers import pipeline

# Try using "mps" for Metal (Mac), "cuda" if you have GPU, and "cpu" if not
device = torch.device("cuda", 0)

pipe = pipeline("automatic-speech-recognition",
      model="NbAiLab/scream_medium_beta",
      chunk_length_s=30,
      device=device,
      max_new_tokens=128,
      generate_kwargs={"language": "norwegian", "task": "transcribe"})

# Load the WAV file. Modify this to use mp3 instead
audio_path = '../media/lrntech_63f9fe069cef060011647ceb.wav'
samples, sample_rate = librosa.load(audio_path, sr=16000, mono=True)

# Run the pipeline
prediction = pipe(samples)["text"]

print(prediction)



In [ ]:
prediction

In [4]:
import requests
#SERVER_URL = "http://127.0.0.1:8008"
SERVER_URL = "http://192.168.2.239"
# gather the podcast slug, episode guid, and audio file link for transcription
podcasts = requests.get(f"{SERVER_URL}/api/podcasts/")
podcasts = podcasts.json()
for i, p in enumerate(podcasts):
    print(f"{i}: {p['title']} ({p['slug']})")

0: Verdict with Ted Cruz (verdict-with-ted-cruz)
1: Leger om livet (leger-om-livet)
2: Norsken, svensken og dansken (norsken-svensken-og-dansken)
3: Huberman Lab (huberman-lab)
4: The Ben Shapiro Show (the-ben-shapiro-show)
5: The Megyn Kelly Show (the-megyn-kelly-show)
6: The Daily (the-daily)
7: Checks and Balance from The Economist (checks-and-balance-from-the-economist)
8: Pod Save America (pod-save-america)
9: Logbuch:Netzpolitik (logbuchnetzpolitik)
10: Apokalypse & Filterkaffee (apokalypse-filterkaffee)
11: Lage der Nation - der Politik-Podcast aus Berlin (lage-der-nation-der-politik-podcast-aus-berlin)
12: Inside Europe | Deutsche Welle (inside-europe-deutsche-welle)
13: Forklart (forklart)
14: Oppdatert (oppdatert)
15: USApodden (usapodden)
16: Det Store Bildet (det-store-bildet)
17: Best of the Left - Progressive Politics and Culture, Curated by Humans, Not Algorithms (best-of-the-left-progressive-politics-and-culture)
18: Gaslit Nation with Andrea Chalupa and Sarah Kendzior 

In [10]:
# convert time in seconds to time in hours, minutes, seconds, 
#including leading zeros and rounding seconds
def convert_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = round(seconds % 60)
    return "{:02d}:{:02d}:{:02d}".format(hours, minutes, seconds)


In [15]:
podcast = podcasts[36]

In [16]:
import datetime
import time
import torch
import numpy as np
import librosa
from transformers import pipeline

for ep in podcast.get("audioitem_set"):
    print(f"{podcast.get('slug')}_{ep.get('guid')}.wav")

    transcription_dict = {}
    transcription_dict["name"] = "wav2vec"
    transcription_dict["words"] = ""
    transcription_dict["speech2txt"] = {
        "model": "scream-med-be-hf",
        "size": "medium",
        "chunk_length_s": 30,
    },
    transcription_dict["language"] = "no"
    transcription_dict["diarization"] = ""
    transcription_dict["guid"] = ep.get('guid')

    # Try using "mps" for Metal (Mac), "cuda" if you have GPU, and "cpu" if not
    start_time = time.time()
    device = torch.device("cuda:0")

    pipe = pipeline("automatic-speech-recognition",
        model="NbAiLab/scream_medium_beta",
        device=device,
        )

    # change language as required
    #pipe.model.config.forced_decoder_ids = pipe.tokenizer.get_decoder_prompt_ids(language="Norwegian", task="transcribe")
    # Load the WAV file. Modify this to use mp3 instead
    audio_path = f"../media/{podcast.get('slug')}_{ep.get('guid')}.wav" 
    samples, sample_rate = librosa.load(audio_path, sr=16000, mono=True)

    # Run the pipeline
    prediction = pipe(samples)["text"]
    running_time = time.time() - start_time

    transcription_dict["text"] = prediction
    transcription_dict['runtime'] = convert_time(running_time)
    transcription_dict['created'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # post transcription to api
    res = requests.post(f"{SERVER_URL}/api/transcriptions/", json=transcription_dict)
    print(res.status_code)





lrntech_647b70d05f71ea00115e25fd.wav


/home/adamj/anaconda3/envs/whspr/lib/python3.8/site-packages/transformers/generation/utils.py:1313: UserWarning: Using `max_length`'s default (448) to control the generation length. This behaviour is deprecated and will be removed from the config in v5 of Transformers -- we recommend using `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


201
lrntech_647b6f8d7b38dd00119f44c3.wav
201
lrntech_647b6e79c1e93b0011ade23c.wav
201
lrntech_647b6d0ab771bd00119196df.wav


KeyboardInterrupt: 

In [23]:
import re
import whisper
import torch

def hf_to_whisper_states(text):
    text = re.sub('.layers.', '.blocks.', text)
    text = re.sub('.self_attn.', '.attn.', text)
    text = re.sub('.q_proj.', '.query.', text)
    text = re.sub('.k_proj.', '.key.', text)
    text = re.sub('.v_proj.', '.value.', text)
    text = re.sub('.out_proj.', '.out.', text)
    text = re.sub('.fc1.', '.mlp.0.', text)
    text = re.sub('.fc2.', '.mlp.2.', text)
    text = re.sub('.fc3.', '.mlp.3.', text)
    text = re.sub('.fc3.', '.mlp.3.', text)
    text = re.sub('.encoder_attn.', '.cross_attn.', text)
    text = re.sub('.cross_attn.ln.', '.cross_attn_ln.', text)
    text = re.sub('.embed_positions.weight', '.positional_embedding', text)
    text = re.sub('.embed_tokens.', '.token_embedding.', text)
    text = re.sub('model.', '', text)
    text = re.sub('attn.layer_norm.', 'attn_ln.', text)
    text = re.sub('.final_layer_norm.', '.mlp_ln.', text)
    text = re.sub('encoder.layer_norm.', 'encoder.ln_post.', text)
    text = re.sub('decoder.layer_norm.', 'decoder.ln.', text)
    return text

# Load HF Model
hf_state_dict = torch.load("/home/adamj/asr_weights/scream_medium.bin")    # pytorch_model.bin file

# Rename layers
for key in list(hf_state_dict.keys())[:]:
    new_key = hf_to_whisper_states(key)
    hf_state_dict[new_key] = hf_state_dict.pop(key)

# Init Whisper Model and replace model weights
whisper_model = whisper.load_model('medium')
whisper_model.load_state_dict(hf_state_dict)

<All keys matched successfully>

In [24]:
# Init Whisper Model and replace model weights
whisper_model = whisper.load_model('medium')
whisper_model.load_state_dict(hf_state_dict)

<All keys matched successfully>

In [66]:
import time
import datetime

decode_options = dict(best_of=5, beam_size=5, language="no")#episode.get("language"))
#transcribe_options = dict(word_timestamps=True, fp16=False, **decode_options)

transcribe_options = dict(compression_ratio_threshold=1.8, no_speech_threshold=0.4, logprob_threshold=-10000, condition_on_previous_text=False, **decode_options)

# run Whisper
#start_time = time.time()
#output = whisper_model.transcribe("/home/adamj/factcheck-podcasts/src/media/lrntech_63f9fe069cef060011647ceb.mp3", **transcribe_options)
#running_time = time.time() - start_time

#list_of_word_list = [seg_dict.pop("words") for seg_dict in output.get("segments")]
#words = [words for seglist in list_of_word_list for words in seglist]

In [67]:



for ep in podcast.get("audioitem_set"):
    print(f"{podcast.get('slug')}_{ep.get('guid')}.wav")
    audio_path = f"../media/{podcast.get('slug')}_{ep.get('guid')}.wav" 

    transcription_dict = {}
    transcription_dict["name"] = "scream-med-beta"
    transcription_dict["speech2txt"] = {
        "model": "scream-med-beta",
        "size": "medium",
    },
    transcription_dict["language"] = "no"
    transcription_dict["diarization"] = ""
    transcription_dict["guid"] = ep.get('guid')

    start_time = time.time()
    output = whisper_model.transcribe(audio_path, **transcribe_options)
    running_time = time.time() - start_time

    transcription_dict["text"] = output.get("text")
    transcription_dict["words"] = ""#[words for seglist in [seg_dict.pop("words") for seg_dict in output.get("segments")] for words in seglist]
    transcription_dict['runtime'] = convert_time(running_time)
    transcription_dict['created'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # post transcription to api
    res = requests.post(f"{SERVER_URL}/api/transcriptions/", json=transcription_dict)
    print(res.status_code)



lrntech_647b70d05f71ea00115e25fd.wav
201
lrntech_647b6f8d7b38dd00119f44c3.wav


KeyboardInterrupt: 

In [26]:
# load audio and pad/trim it to fit 30 seconds
audio = whisper.load_audio(audio_path)
audio = whisper.pad_or_trim(audio)

# make log-Mel spectrogram and move to the same device as the model
mel = whisper.log_mel_spectrogram(audio).to(whisper_model.device)

# detect the spoken language
_, probs = whisper_model.detect_language(mel)
print(f"Detected language: {max(probs, key=probs.get)}")

# decode the audio
options = whisper.DecodingOptions()
result = whisper.decode(whisper_model, mel, options)

# print the recognized text
print(result.text)

Detected language: no
Hei og velkommen til LØRN. 1500 læringshistorier fra de beste fremtidstenkerne og skaperne. På lorn.tek kan du lytte, se eller lese alt innhold gratis, men registrer deg for å få tilgang til personlige læringstider, sertifikater og mye mer. Hei og velkommen til LØRN-serien med BI om compliance.


In [27]:
test = whisper_model.transcribe(audio_path, language='no')

In [29]:
test.get("text")

' Hei og velkommen til LØRN. 1500 læringshistorier fra de beste fremtidstenkerne og skaperne. På lorn.tek kan du lytte, se eller lese alt innhold gratis, men registrer deg for å få tilgang til personlige læringstider, sertifikater og mye mer. Hei og velkommen til LØRN-serien med BI om compliance. Vi i LØRN skal i samtaler med Milos Novic fra BI. Flere gjester skal diskutere de viktigste driverne for compliance og GDPR. Velkommen til dere, Frode Skårnes og Milos Novovic. Tusen takk. Takk så mye. Jeg vil gjøre lytteren oppmerksom på at jeg sier etternavnet ditt helt riktig. Jeg får den ikke sant. Vi kan ikke drive med det hos Novovic. Det er helt riktig. Dette er en ekstra koselig samtale for meg. Som eksperten på deg, Sikkerhet, har vi med oss vår egen Frode Skårnes, en av partnerne i LØRN. Velkommen til dere to. Jeg kan ikke komme på med noe nytt, bortsetter fra å si at jeg jobber med jus og tek. Det er veldig vanskelig. Favorittbandet fra gamlelandet? Hvorfor er det vanskelig å si? De

In [ ]:
ep = None
for episode in podcast.get("audioitem_set"):
    if episode.get("guid") == "63f9fe069cef060011647ceb":
        ep = episode
        break

In [ ]:
transcription_dict = {}
transcription_dict["name"] = "wav2vec"
transcription_dict["words"] = ""
transcription_dict["speech2txt"] = {
    "model": "TEST WHISPERnbai",
},
transcription_dict["language"] = "no"
transcription_dict["diarization"] = ""
transcription_dict["guid"] = ep.get('guid')

transcription_dict["text"] = output.get("text")
transcription_dict['runtime'] = convert_time(running_time)
transcription_dict['created'] = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# post transcription to api
res = requests.post(f"{SERVER_URL}/api/transcriptions/", json=transcription_dict)
print(res.status_code)


In [ ]:
output.get("text")

In [ ]:
std(output.get("text"))

In [ ]:
from whisper.normalizers import BasicTextNormalizer
whisper_normalizer = BasicTextNormalizer()